# **TESINA: Sequía Forestal**



# **Objetivo Específico 1**

## Recursos y Librerías


In [1]:
# Recurso para conocer porcentaje de datos Nan en las variables

!git clone https://github.com/jsblandon/weather_data_uy_preprocessing.git
import sys
sys.path.append('/content/weather_data_uy_preprocessing')

Cloning into 'weather_data_uy_preprocessing'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 25 (delta 11), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 21.37 KiB | 4.27 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [ ]:
# Importación de librerías

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.axes._axes import _log as matplotlib_axes_logger
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from weather_data_preprocessing import null_report
import scipy.stats as stats
import glob
import os

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

sns.set_style('whitegrid')

In [ ]:
import matplotlib.dates as mdates

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Conectar DRIVE para cargar datos

from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

##Carga de Datos de Telepluviómetros INUMET

In [ ]:
# Se cargan los datos de INUMET
# Carpeta con archivos .csv telepluviometros

# Ruta Sofi
carpeta = '/content/drive/MyDrive/Tesina/Telepluviometros'

# Ruta Nelcy
#carpeta = '/content/drive/MyDrive/Investigacion/Tutoria_Tesis_TrabajosGrado/Sequias_forestal/Datos Inumet/Telepluviómetros'

# Ruta Fran
#carpeta = '/content/drive/MyDrive/Tesina/Telepluviometros'

# Lista con los archivos dentro de la carpeta
csv_archivos = [file for file in os.listdir(carpeta) if file.endswith('.csv')]

# Inicializar listas para los DataFrames y nombres relevantes
dfs = []
nombres = []

# Cargar los archivos y procesarlos
for file in csv_archivos:
    archivo = os.path.join(carpeta, file)
    nombre_relevante = file.split('_')[5]  # Extrae el nombre relevante
    nombres.append(nombre_relevante)

    # Leer el archivo y procesar el DataFrame
    df = pd.read_csv(archivo)
    df['rtuTimestamp'] = pd.to_datetime(df['rtuTimestamp']).dt.tz_convert(None)
    df.set_index('rtuTimestamp', inplace=True)
    df.sort_index(inplace=True)

    # Eliminar la columna rtuBattery si existe
    if 'rtuBattery' in df.columns:
        df.drop(columns=['rtuBattery'], inplace=True)

    dfs.append(df)

# Definir rango de tiempo continuo con una frecuencia de 5 minutos
start_date = '2022-12-21'
end_date = '2023-12-21'
rango_fechas = pd.date_range(start=start_date, end=end_date, freq='5min')

# Reindexar cada DataFrame con el rango continuo y renombrar columna
dfs_reindexados = [
    df.reindex(rango_fechas).rename(columns={'rtuRain': nombre})
    for df, nombre in zip(dfs, nombres)
]

# Concatenar los DataFrames reindexados
inumet = pd.concat(dfs_reindexados, axis=1)

# Calcular valores faltantes
datos_faltantes = inumet.isna().sum()

datos_faltantes

# Calcular porcentaje de datos faltantes
porcentaje_faltantes = (inumet.isna().sum() / len(inumet)) * 100

# Combinar cantidad y porcentaje en un solo DataFrame
datos_faltantes = pd.DataFrame({
    'faltantes': inumet.isna().sum(),
    'porcentaje': porcentaje_faltantes
})

# Mostrar los datos faltantes con porcentaje
print(datos_faltantes)

In [ ]:
# Quebracho muestra un porcentaje de 31% por lo que esta estacion será eliminada.
# Las demás estaciones se encuentran entre 0.6 y 1,5 %, por lo que podemos continuar con ellas.

##Carga de Datos de CHIRPS

In [ ]:
# Se cargan los datos de CHIRPS
# Carpeta con archivos .csv chirps

# Ruta Francisco
carpeta = '/content/drive/MyDrive/Tesina/Datos_CHIRPS'

# Ruta Nelcy
#carpeta = '/content/drive/MyDrive/Investigacion/Tutoria_Tesis_TrabajosGrado/Sequias_forestal/Datos_CHIRPS'

# Ruta Sofi
# carpeta = '/content/drive/MyDrive/Tesina/Datos_CHIRPS'

# Lista con los archivos dentro de la carpeta
csv_archivos = [file for file in os.listdir(carpeta) if file.endswith('.csv')]

# Función para generar el reporte de valores NaN
def null_report(df):
    return df.isnull().mean() * 100

# Crear una lista para almacenar la información del resumen
data_summary = []

# For para operar sobre los archivos
for file in csv_archivos:
    archivo = os.path.join(carpeta, file)
    # Extraemos el nombre relevante antes de "(TP)" y eliminamos los guiones bajos y convertimos a minúsculas
    nombre_relevante = file.split('(TP)')[0].replace('_', '').lower()
    df = pd.read_csv(archivo)

    # Convertir a datetime
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    df.sort_values(by='date', inplace=True)

    # Crear una variable en el entorno global con el nombre relevante
    globals()[nombre_relevante] = df

    # Obtener fechas de inicio y finalización
    fecha_inicio = df.index.min()
    fecha_fin = df.index.max()

    # Obtener porcentajes de valores NaN
    nans = null_report(df)

    # Agregar información al resumen
    summary = {
        'Nombre Relevante': nombre_relevante,
        'Fecha Inicio': fecha_inicio,
        'Fecha Fin': fecha_fin
    }
    # Añadir columnas con NaN al resumen
    for col, pct_nan in nans.items():
        summary[f'% NaN en {col}'] = pct_nan

    data_summary.append(summary)

# Crear un DataFrame con el resumen
df_summary = pd.DataFrame(data_summary)

# Mostrar el DataFrame resumen
print(df_summary)

In [ ]:
# Los datos CHIRPS no presentan datos faltantes

In [ ]:
# Crear un solo df con todos los datos CHRIRPS del 2022 al 2023 para la validación

# Lista con DataFrames
dfs = [paradorlavibora, piedrasola, sarandidelnavarro, sanjavier, chapicuy, piedrascoloradas, elaguila, pueblogrecco, quebracho, guichon, nuevoberlin, eleucaliptus]

# Nombres de cada DataFrame
nombres = ['paradorlavibora', 'piedrasola', 'sarandidelnavarro', 'sanjavier', 'chapicuy', 'piedrascoloradas', 'elaguila', 'pueblogrecco', 'quebracho', 'guichon', 'nuevoberlin', 'eleucaliptus']

# Convertimos el índice a datetime, filtramos el rango de fechas y seleccionamos solo 'precipitation'
dfs_filtrados = [
    df.loc[(df.index >= '2022-12-21') & (df.index <= '2023-12-21'), ['precipitation']].rename(columns={'precipitation': nombre})
    for df, nombre in zip(dfs, nombres)
]

# Unimos todos los DataFrames en uno solo basado en el índice de fecha
chirps = pd.concat(dfs_filtrados, axis=1)

# Mostrar el DataFrame unificado
chirps

# Análisis Exploratorio INUMET



## Análisis Cincominutal de las Variables

In [ ]:
inumet

In [ ]:
inumet.describe()

In [ ]:
# Datos faltantes
nans_inumet = null_report(inumet)
print(nans_inumet)

In [ ]:
inumet_sin_quebracho = inumet.drop('quebracho', axis=1)

In [ ]:
inumet_sin_quebracho.describe()

In [ ]:
import matplotlib.dates as mdates

# Calcular porcentaje de datos faltantes por día
datos_por_dia = inumet.resample('D').count()
porcentaje_faltantes = (1 - datos_por_dia / 288) * 100

# Preparar colores y estilos
num_series = len(porcentaje_faltantes.columns)
colormap = plt.cm.tab10  # Colores clásicos y suaves
colors = [colormap(i % 10) for i in range(num_series)]
estilos = ['-', '--', '-.', ':']

# Filtrar días con >10% faltantes
porcentaje_filtrado = porcentaje_faltantes.mask(porcentaje_faltantes > 10, np.nan)

# Crear figura con subplots verticales
fig, axs = plt.subplots(2, 1, figsize=(14, 10), sharex=True, sharey=True)

# Formato de fecha mm-yy
date_format = mdates.DateFormatter("%m-%y")

# ---------- Subplot 1: Todos los datos ----------
for i, columna in enumerate(porcentaje_faltantes.columns):
    axs[0].plot(
        porcentaje_faltantes.index,
        porcentaje_faltantes[columna],
        label=columna,
        color=colors[i % len(colors)],
        linestyle=estilos[i % len(estilos)],
        linewidth=1.5
    )

axs[0].set_title("Datos Cincominutales", fontsize=18, fontweight='bold')
axs[0].set_ylabel("Faltantes por día (%)", fontsize=16, fontweight='bold')
axs[0].tick_params(axis='both', labelsize=16)
axs[0].xaxis.set_major_formatter(date_format)  # aplicar formato fecha
axs[0].grid(True, axis='y', linestyle='--', alpha=0.7)

# Subplot 2: Filtrado ≤10% (sin quebracho)
columnas_sin_quebracho = [col for col in porcentaje_filtrado.columns if col.lower() != 'quebracho']

for i, columna in enumerate(columnas_sin_quebracho):
    axs[1].plot(
        porcentaje_filtrado.index,
        porcentaje_filtrado[columna],
        label=columna,
        color=colors[i % len(colors)],
        linestyle=estilos[i % len(estilos)],
        linewidth=1.5
    )

axs[1].set_title("Datos Cincominutales Filtrados", fontsize=18, fontweight='bold')
axs[1].set_xlabel("Fecha", fontsize=16, fontweight='bold')
axs[1].set_ylabel("Faltantes por día (%)", fontsize=16, fontweight='bold')
axs[1].tick_params(axis='both', labelsize=16)
axs[1].xaxis.set_major_formatter(date_format)  # aplicar formato fecha
axs[1].legend(
    title="Estaciones",
    loc="upper right",
    fontsize=13,
    title_fontsize=14,
    frameon=True
)
axs[1].grid(True, axis='y', linestyle='--', alpha=0.7)

# Ajustar diseño
plt.tight_layout()
plt.savefig("31.png", dpi=300)
plt.show()

In [ ]:
# Fechas de datos faltantes

# Cantidad total de datos esperados por día
total_datos_dia = 288

# Calcular cantidad de datos faltantes diarios por estación
faltantes_por_dia = total_datos_dia - datos_por_dia

# Crear DataFrame con porcentaje y cantidad de faltantes
df_faltantes = porcentaje_faltantes.copy()
df_faltantes['fecha'] = porcentaje_faltantes.index

# Convertir a formato largo para facilitar filtrado y visualización
df_largo = df_faltantes.melt(id_vars='fecha', var_name='estacion', value_name='porc_faltantes')

# Agregar columna de cantidad de datos faltantes
df_largo['cant_faltantes'] = df_largo['porc_faltantes'] * total_datos_dia / 100

# Filtrar filas donde porcentaje de faltantes > 10%
df_periodos = df_largo[df_largo['porc_faltantes'] > 10].copy()

# Ordenar por estación y fecha
df_periodos.sort_values(['estacion', 'fecha'], inplace=True)

# Mostrar tabla resultado
df_periodos[['estacion', 'fecha', 'porc_faltantes', 'cant_faltantes']]

In [ ]:
# Asegurar que el índice es datetime
inumet_sin_quebracho.index = pd.to_datetime(inumet_sin_quebracho.index)

# Calcular el porcentaje de NaNs por día y columna
porcentaje_nans_por_dia = inumet_sin_quebracho.isna().resample('D').mean()

# Identificar los días que tienen >10% de NaNs en alguna columna
dias_con_muchos_nans = porcentaje_nans_por_dia.max(axis=1) > 0.10

# Obtener los días a eliminar
dias_a_eliminar = porcentaje_nans_por_dia.index[dias_con_muchos_nans]

# Eliminar del DataFrame todos los registros correspondientes a esos días
inumet_final = inumet_sin_quebracho[~inumet_sin_quebracho.index.floor('D').isin(dias_a_eliminar)]

In [ ]:
inumet_final.describe()

In [ ]:
# Calcular los días con más del 20% de datos faltantes por estación
dias_superan_20 = (porcentaje_faltantes > 99).sum()
print("Días con más del 20% de datos faltantes por estación:")
print(dias_superan_20)

# Configurar los subplots en una cuadrícula de 3x4
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 12), sharex=True)
axes = axes.flatten()  # Asegurar que los ejes estén en un solo arreglo

# Crear un subplot para cada estación
for i, columna in enumerate(porcentaje_faltantes.columns):
    ax = axes[i]
    ax.plot(porcentaje_faltantes.index, porcentaje_faltantes[columna], label=columna, color='b')
    ax.set_title(f"{columna}", fontsize=12)
    ax.set_ylabel("% Faltantes", fontsize=10)
    ax.grid(True)
    ax.legend(loc="upper right", fontsize=8)

# Eliminar subplots vacíos si las estaciones no llenan toda la cuadrícula
for j in range(len(porcentaje_faltantes.columns), len(axes)):
    fig.delaxes(axes[j])

# Configurar el eje X y ajustar el diseño
plt.xlabel("Fecha", fontsize=12)
plt.tight_layout()

# Guardar y mostrar el gráfico
plt.savefig("inumet_nansindividual.jpg")
plt.show()

In [ ]:
fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

min_val = inumet.min().min()
max_val = inumet.max().max()

for ax in axs:
    ax.tick_params(axis='both', labelsize=12)

max_freq = 0
for columna in inumet.columns:
    datos_filtrados = inumet[columna].dropna()
    _, bins = np.histogram(datos_filtrados, bins=30)
    hist, _ = np.histogram(datos_filtrados, bins=bins)
    max_freq = max(max_freq, hist.max())

for i, columna in enumerate(inumet.columns):
    datos_filtrados = inumet[columna].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(min_val, max_val)
    axs[i].set_ylim(0, max_freq)

# Ejes generales en lugar de títulos individuales
fig.supxlabel("Precipitación (mm)", fontsize=14, fontweight='bold')
fig.supylabel("Frecuencia", fontsize=14, fontweight='bold')

fig.suptitle("Histogramas INUMET Datos Cincominutales", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig("hist-cin-1.jpg")
plt.show()

In [ ]:
# Histograma Cincominutal SIN CEROS

fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

# Rango sin ceros
min_val = inumet[inumet != 0].min().min()
max_val = inumet[inumet != 0].max().max()

# Tamaño de ticks en todos los subplots
for ax in axs:
    ax.tick_params(axis='both', labelsize=12)

# Determinar frecuencia máxima
max_freq = 0
for columna in inumet.columns:
    datos_filtrados = inumet[columna][inumet[columna] != 0].dropna()
    _, bins = np.histogram(datos_filtrados, bins=30)
    hist, _ = np.histogram(datos_filtrados, bins=bins)
    max_freq = max(max_freq, hist.max())

# Dibujar histogramas
for i, columna in enumerate(inumet.columns):
    datos_filtrados = inumet[columna][inumet[columna] != 0].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(min_val, max_val)
    axs[i].set_ylim(0, max_freq)

# Ejes generales
fig.supxlabel("Precipitación (mm)", fontsize=14, fontweight='bold')
fig.supylabel("Frecuencia", fontsize=14, fontweight='bold')

# Título general
fig.suptitle("Histogramas INUMET Cincominutal (Sin 0s)", fontsize=16, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("hist-cin-2.jpg")
plt.show()

In [ ]:
# Boxplot Inumet Cincominutal SIN CEROS

# Filtrar valores > 0.1 y eliminar NaN por estación (columna)
datos_filtrados = [inumet[columna][inumet[columna] > 0.1].dropna() for columna in inumet.columns]

# Crear la figura del boxplot
fig, ax = plt.subplots(figsize=(12, 7))
ax.boxplot(datos_filtrados, vert=True, patch_artist=True,
           boxprops=dict(facecolor='skyblue', color='black'))

# Personalización unificada
ax.set_title("Diagramas de caja INUMET Cincominutal (Sin 0s)", fontsize=16, fontweight='bold')
ax.set_ylabel("Precipitación (mm)", fontsize=14, fontweight='bold')
ax.set_xlabel("Estaciones", fontsize=14, fontweight='bold')
ax.set_xticklabels(inumet.columns, rotation=45, ha='right', fontsize=12, fontweight='bold')

# Ticks
ax.tick_params(axis='both', labelsize=12)

# Grilla
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("box-cin-1.jpg")
plt.show()

In [ ]:
# Serie Temporal INUMET Cincominutal

# Calcular el valor mínimo y máximo de todas las estaciones
y_min = inumet.min().min()
y_max = inumet.max().max()

# Definir el rango de fechas
inicio0 = inumet.index.min()
fin0 = pd.Timestamp('2023-12-21')  # Fecha máxima a mostrar

# Crear la cuadrícula de subplots 4x3
fig, axes = plt.subplots(4, 3, figsize=(16, 12))
axes = axes.flatten()

# Formato de fecha mm-yy (últimos dos dígitos del año)
date_format = mdates.DateFormatter("%m-%y")

# Crear un gráfico de líneas azules para cada estación
for i, columna in enumerate(inumet.columns):
    axes[i].plot(inumet.index, inumet[columna], label=columna, color='steelblue', linewidth=1.2)
    axes[i].set_title(f'{columna}', fontsize=16)  # antes 14 → ahora 16
    axes[i].set_ylim(y_min, y_max)
    axes[i].set_xlim(inicio0, fin0)
    axes[i].grid(True, axis='y', linestyle='--', alpha=0.7)

    # Aplicar formato de fecha
    axes[i].xaxis.set_major_formatter(date_format)

    # Tamaño de ticks
    axes[i].tick_params(axis='both', labelsize=14)  # antes 12 → ahora 14

# Eliminar subplots vacíos si hay menos de 12 estaciones
if len(inumet.columns) < len(axes):
    for j in range(len(inumet.columns), len(axes)):
        fig.delaxes(axes[j])

# Ejes generales
fig.supxlabel("Fecha", fontsize=16, fontweight='bold')       # antes 14 → 16
fig.supylabel("Precipitación (mm)", fontsize=16, fontweight='bold')  # antes 14 → 16

# Título general
fig.suptitle("Serie Temporal INUMET Cincominutal", fontsize=18, fontweight='bold')  # antes 16 → 18

# Ajustar el diseño
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar figura
plt.savefig("serie_temporal_inumet_cincominutal.jpg", dpi=300)

# Mostrar figura
plt.show()

##Análisis Diario de las Variables

In [ ]:
# Cambiamos la frecuencia a diaria sumando las precipitaciones de cada día
inumet_diario_completo = inumet_final.resample('D').sum()

# Conservar solo los días que efectivamente existían en el índice original
dias_validos = inumet_final.index.floor('D').unique()
inumet_diario = inumet_diario_completo.loc[inumet_diario_completo.index.isin(dias_validos)]

In [ ]:
inumet_diario.describe()

In [ ]:
# Datos faltantes
nans_inumet_diario = null_report(inumet_diario)
print(nans_inumet_diario)

In [ ]:
#Histograma INUMET diario con CERO

fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

min_val = inumet_diario.min().min()
max_val = inumet_diario.max().max()

max_freq = 0
for columna in inumet_diario.columns:
    datos_filtrados = inumet_diario[columna].dropna()
    _, bins = np.histogram(datos_filtrados, bins=30)
    hist, _ = np.histogram(datos_filtrados, bins=bins)
    max_freq = max(max_freq, hist.max())

for i, columna in enumerate(inumet_diario.columns):
    datos_filtrados = inumet_diario[columna].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].set_xlabel('Precipitación (mm)', fontsize=9, fontweight='bold')
    axs[i].set_ylabel('Frecuencia', fontsize=9, fontweight='bold')
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(min_val, max_val)
    axs[i].set_ylim(0, max_freq)

fig.suptitle("Histogramas INUMET Datos Diarios", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Histograma INUMET Diario SIN CEROS

fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

# Límites fijos
x_min, x_max = 0, 125
y_min, y_max = 0, 60

# Tamaño de ticks en todos los subplots
for ax in axs:
    ax.tick_params(axis='both', labelsize=12)

# Determinar frecuencia máxima
max_freq = 0
for columna in inumet_diario.columns:
    datos_filtrados = inumet_diario[columna][inumet_diario[columna] != 0].dropna()
    _, bins = np.histogram(datos_filtrados, bins=30)
    hist, _ = np.histogram(datos_filtrados, bins=bins)
    max_freq = max(max_freq, hist.max())

# Dibujar histogramas
for i, columna in enumerate(inumet_diario.columns):
    datos_filtrados = inumet_diario[columna][inumet_diario[columna] != 0].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(x_min, x_max)
    axs[i].set_ylim(y_min, y_max)

# Eliminar subplot sobrante si hay menos estaciones
if len(inumet_diario.columns) < len(axs):
    for j in range(len(inumet_diario.columns), len(axs)):
        fig.delaxes(axs[j])

# Ejes generales
fig.supxlabel("Precipitación (mm)", fontsize=14, fontweight='bold')
fig.supylabel("Frecuencia", fontsize=14, fontweight='bold')

# Título general
fig.suptitle("Histogramas INUMET Datos Diarios (Sin 0s)", fontsize=16, fontweight='bold')

# Ajustar diseño
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("inumet_hist_diariosin0.jpg", dpi=300)
plt.show()

In [ ]:
# Minimos por estación en datos diarios
minimos = inumet_diario.min()
print(minimos)

In [ ]:
# Mínimos por estación distintos de cero en datos diarios
minimos_sin_cero = inumet_diario[inumet_diario != 0].min()
print(minimos_sin_cero)

In [ ]:
# Máximos por estación en datos diarios
maximos = inumet_diario.max()
print(maximos)

In [ ]:
import matplotlib.dates as mdates

# Series Temporales Inumet Diario

# Configuración de la cuadrícula
fig, axs = plt.subplots(4, 3, figsize=(12, 10))
axs = axs.flatten()

# Definir el límite máximo de fecha
inicio1 = inumet_diario.index.min()
fin1 = pd.Timestamp('2023-12-21')

# Crear un gráfico de líneas para cada estación
for i, columna in enumerate(inumet_diario.columns):
    axs[i].plot(inumet_diario.index, inumet_diario[columna], label=columna, color='steelblue')
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].set_ylim(0, 125)
    axs[i].set_xlim(inicio1, fin1)
    axs[i].grid(True, axis='y', linestyle='--', alpha=0.7)

    # Ticks con labelsize uniforme
    axs[i].tick_params(axis='both', which='major', labelsize=12)

    # Formato de fechas en mm-yy
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%m-%y'))

# Eliminar subplots vacíos si hay menos estaciones que paneles
if len(inumet_diario.columns) < len(axs):
    for j in range(len(inumet_diario.columns), len(axs)):
        fig.delaxes(axs[j])

# Etiquetas generales de los ejes (en vez de repetir en cada subplot)
fig.supxlabel("Fecha", fontsize=14, fontweight='bold')
fig.supylabel("Precipitación (mm)", fontsize=14, fontweight='bold')

# Título general del gráfico
fig.suptitle("Series Temporales INUMET diario", fontsize=16, fontweight='bold')

# Ajuste de espaciado
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar figura
plt.savefig("series_temporales_inumet_diario.png", dpi=300)

# Mostrar figura
plt.show()

In [ ]:
# Series Temporales Inumet Diario

# Configuración de la cuadrícula
fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

# Obtener los límites del eje Y para todas las estaciones
y_min = inumet_diario.min().min()
y_max = inumet_diario.max().max()

# Definir el límite máximo de fecha
inicio1 = inumet_diario.index.min()
fin1 = pd.Timestamp('2023-12-21')

# Crear un gráfico de líneas para cada estación
for i, columna in enumerate(inumet_diario.columns):
    axs[i].plot(inumet_diario.index, inumet_diario[columna], label=columna, color='steelblue')
    axs[i].set_title(f'{columna}', fontsize=16)
    axs[i].set_xlabel('Fecha', fontsize=11, fontweight='bold')
    axs[i].set_ylabel('Precipitación (mm)', fontsize=11, fontweight='bold')
    axs[i].set_ylim(0, 125)
    axs[i].set_xlim(inicio1, fin1)
    axs[i].grid(True, axis='y', linestyle='--', alpha=0.7)

# Eliminar subplots vacíos si hay menos estaciones que paneles
if len(inumet_diario.columns) < len(axs):
    for j in range(len(inumet_diario.columns), len(axs)):
        fig.delaxes(axs[j])

# Título general del gráfico
fig.suptitle("Series Temporales INUMET diario", fontsize=18, fontweight='bold')

# Ajuste de espaciado
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar figura
plt.savefig("series_temporales_inumet_diario.png", dpi=300)

# Mostrar figura
plt.show()

In [ ]:
# Histograma INUMET diario sin CERO
fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

min_val = inumet_diario[inumet_diario != 0].min().min()
max_val = inumet_diario[inumet_diario != 0].max().max()

max_freq = 0
for columna in inumet_diario.columns:
    datos_filtrados = inumet_diario[columna][inumet_diario[columna] != 0].dropna()
    hist, _ = np.histogram(datos_filtrados, bins=30)
    max_freq = max(max_freq, hist.max())

for i, columna in enumerate(inumet_diario.columns):
    datos_filtrados = inumet_diario[columna][inumet_diario[columna] != 0].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'Estación: {columna}', fontsize=14)
    axs[i].set_xlabel('Precipitación (mm)', fontsize=9, fontweight='bold')
    axs[i].set_ylabel('Frecuencia', fontsize=9, fontweight='bold')
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(min_val, 125)
    axs[i].set_ylim(0, 60)

# Eliminar subplots sobrantes
if len(inumet_diario.columns) < len(axs):
    for j in range(len(inumet_diario.columns), len(axs)):
        fig.delaxes(axs[j])

fig.suptitle("Histogramas INUMET Datos Diarios (Sin 0s)", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("inumet_hist_diariosin0.jpg", dpi=300)
plt.show()

In [ ]:
#Boxplot Diario SIN CERO

import matplotlib.pyplot as plt

# Filtrar valores distintos de 0 y eliminar NaN por estación (columna)
datos_filtrados = [inumet_diario[col][inumet_diario[col] != 0].dropna() for col in inumet_diario.columns]

# Crear la figura del boxplot
fig, ax = plt.subplots(figsize=(12, 7))
ax.boxplot(datos_filtrados, vert=True, patch_artist=True,
           boxprops=dict(facecolor='skyblue', color='black'))

# Personalización
ax.set_title("Diagramas de caja INUMET Diario (Sin 0s)", fontsize=16, fontweight='bold')
ax.set_ylabel("Precipitación (mm)", fontsize=10, fontweight='bold')
ax.set_xlabel("Estaciones", fontsize=9, fontweight='bold')
ax.set_xticklabels(inumet_diario.columns, rotation=45, ha='right', fontsize=10, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

##Análisis Mensual de las variables

In [ ]:
# Definir los períodos mensuales del 21 de un mes al 21 del siguiente
periodos = [
    ("2022-12-21", "2023-01-21"),
    ("2023-01-21", "2023-02-21"),
    ("2023-02-21", "2023-03-21"),
    ("2023-03-21", "2023-04-21"),
    ("2023-04-21", "2023-05-21"),
    ("2023-05-21", "2023-06-21"),
    ("2023-06-21", "2023-07-21"),
    ("2023-07-21", "2023-08-21"),
    ("2023-08-21", "2023-09-21"),
    ("2023-09-21", "2023-10-21"),
    ("2023-10-21", "2023-11-21"),
    ("2023-11-21", "2023-12-21")
]

# Crear un DataFrame vacío para almacenar los resultados
inumet_mensual = pd.DataFrame()

# Calcular la precipitación acumulada para cada período
for inicio, fin in periodos:
    # Seleccionar los datos dentro del intervalo específico y sumar las precipitaciones por columna
    acumulado_periodo = inumet_diario.loc[inicio:fin].sum()  # Sumar por cada columna (punto de medición)

    # Agregar los datos acumulados al DataFrame de resultados
    inumet_mensual[f"{inicio} - {fin}"] = acumulado_periodo

# Transponer el DataFrame para tener los períodos como filas
inumet_mensual = inumet_mensual.T

# Crear una columna con las fechas iniciales del período
inumet_mensual["fecha_inicio"] = [inicio for inicio, fin in periodos]

# Reorganizar las columnas para que "fecha_inicio" esté al principio
inumet_mensual = inumet_mensual[["fecha_inicio"] + [col for col in inumet_mensual.columns if col != "fecha_inicio"]]

# Convertir la columna 'fecha_inicio' a datetime
inumet_mensual['fecha_inicio'] = pd.to_datetime(inumet_mensual['fecha_inicio'], format='%Y-%m-%d')

# Configurar 'fecha_inicio' como índice del DataFrame
inumet_mensual.set_index('fecha_inicio', inplace=True)

# Ordenar el DataFrame por el índice temporal
inumet_mensual.sort_index(inplace=True)

# Mostrar el DataFrame resultante
inumet_mensual

In [ ]:
inumet_mensual.describe()

In [ ]:
# Datos faltantes
nans_inumet_mensual = null_report(inumet_mensual)
print(nans_inumet_mensual)

In [ ]:
#Histogramas INUMET mensual con CERO

fig, axs = plt.subplots(4, 3, figsize=(12.25, 10))
axs = axs.flatten()

# Límites comunes para ejes
xmin = inumet_mensual.min().min()
xmax = inumet_mensual.max().max()
ymax = max(inumet_mensual[col].value_counts().max() for col in inumet_mensual.columns)

for i, columna in enumerate(inumet_mensual.columns):
    datos_filtrados = inumet_mensual[columna].dropna()
    axs[i].hist(datos_filtrados, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].set_xlabel('Precipitación (mm)', fontsize=10, fontweight='bold')
    axs[i].set_ylabel('Frecuencia', fontsize=10, fontweight='bold')
    axs[i].set_xlim(xmin, xmax)
    axs[i].set_ylim(0, ymax)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)

fig.suptitle("Histogramas INUMET Datos Mensuales", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
#Histogramas INUMET mensual SIN CERO

fig, axs = plt.subplots(4, 3, figsize=(12.25, 10))
axs = axs.flatten()

# Cálculo de límites comunes SIN ceros
xmin = inumet_mensual[inumet_mensual > 0.1].min().min()
xmax = inumet_mensual[inumet_mensual > 0.1].max().max()
ymax = 0

# Calcular frecuencia máxima para escala uniforme
for col in inumet_mensual.columns:
    datos_filtrados = inumet_mensual[col][inumet_mensual[col] > 0.1].dropna()
    hist, _ = np.histogram(datos_filtrados, bins=20)
    ymax = max(ymax, hist.max())

# Graficar
for i, columna in enumerate(inumet_mensual.columns):
    datos_filtrados = inumet_mensual[columna][inumet_mensual[columna] > 0.1].dropna()
    axs[i].hist(datos_filtrados, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].set_xlabel('Precipitación (mm)', fontsize=10, fontweight='bold')
    axs[i].set_ylabel('Frecuencia', fontsize=10, fontweight='bold')
    axs[i].set_xlim(xmin, xmax)
    axs[i].set_ylim(0, ymax)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)

fig.suptitle("Histogramas INUMET Datos Mensuales (Sin 0s)", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Minimos por estación en datos mensuales
minimos_m = inumet_mensual.min()
print(minimos_m)

In [ ]:
#Gráficos ordenados por ISOYETAS

# Define el orden deseado de las estaciones
estaciones_ordenadas = ['piedrasola', 'eleucaliptus', 'chapicuy',
                        'sarandidelnavarro', 'guichon',
                        'piedrascoloradas', 'pueblogrecco',
                        'aguila', 'sanjavier', 'nuevoberlin', 'lavibora']

# Reorganiza las columnas del DataFrame según este orden
inumet_mensual_ordenado = inumet_mensual[estaciones_ordenadas]

# Definir colores por grupo de estaciones
colores = {
    'grupo1': 'darkblue',    # Piedrasola, Eleucaliptus, Chapicuy
    'grupo2': 'blue',   # Sarandi del Navarro, Guichon, Quebracho, Piedras Coloradas, Pueblo Grecco
    'grupo3': 'skyblue'   # Aguila, San Javier, Nuevo Berlin, La Vibora
}

# Mapear estaciones a sus colores
colores_estaciones = []
for estacion in estaciones_ordenadas:
    if estacion in ['piedrasola', 'eleucaliptus']:
        colores_estaciones.append(colores['grupo1'])
    elif estacion in ['chapicuy', 'sarandidelnavarro', 'guichon', 'piedrascoloradas', 'pueblogrecco', 'aguila']:
        colores_estaciones.append(colores['grupo2'])
    else:
        colores_estaciones.append(colores['grupo3'])

# Parámetros para la visualización
num_estaciones = len(inumet_mensual_ordenado.columns)
filas, columnas = 4, 3  # Ajusta según tu número de estaciones
fig, axes = plt.subplots(filas, columnas, figsize=(16, 12))
axes = axes.flatten()  # Convertimos la cuadrícula en un array 1D para iterar fácilmente

# Obtener los límites del eje Y
y_max = inumet_mensual_ordenado.max().max()  # Valor máximo de todas las estaciones
y_min = inumet_mensual_ordenado.min().min()  # Valor mínimo de todas las estaciones

# Graficar barras para cada estación
for i, (columna, color) in enumerate(zip(inumet_mensual_ordenado.columns, colores_estaciones)):
    axes[i].bar(inumet_mensual_ordenado.index, inumet_mensual_ordenado[columna],
                label=columna, color=color, width=20)  # Ajusta el ancho de las barras
    axes[i].set_title(f'Estación: {columna}')
    axes[i].set_xlabel('Fecha')
    axes[i].set_ylabel('Precipitación (mm)')
    axes[i].set_ylim(y_min, y_max)  # Establecer límites uniformes en Y

# Eliminar cualquier subgráfico vacío si hay más espacios que estaciones
for i in range(num_estaciones, len(axes)):
    fig.delaxes(axes[i])

# Ajustar el espaciado entre subgráficos
plt.tight_layout()


plt.savefig("inumet_mensual.jpg")

plt.show()

In [ ]:
# Gráficos ordenados por ISOYETAS Según el OTRO MAPA
estaciones_ordenadas = ['piedrasola', 'eleucaliptus', 'chapicuy',
                        'sarandidelnavarro', 'guichon',
                        'piedrascoloradas', 'pueblogrecco',
                        'aguila', 'sanjavier', 'nuevoberlin', 'lavibora']

# Reorganiza las columnas del DataFrame según este orden
inumet_mensual_ordenado = inumet_mensual[estaciones_ordenadas]

# Definir colores por grupo de estaciones
colores = {
    'grupo1': 'darkblue',    # Piedrasola, Eleucaliptus, Chapicuy
    'grupo2': 'blue',   # Sarandi del Navarro, Guichon, Quebracho, Piedras Coloradas, Pueblo Grecco
    'grupo3': 'skyblue'   # Aguila, San Javier, Nuevo Berlin, La Vibora
}

# Mapear estaciones a sus colores
colores_estaciones = []
for estacion in estaciones_ordenadas:
    if estacion in ['piedrasola', 'eleucaliptus', 'chapicuy']:
        colores_estaciones.append(colores['grupo1'])
    elif estacion in ['sarandidelnavarro', 'guichon', 'piedrascoloradas', 'pueblogrecco', 'aguila']:
        colores_estaciones.append(colores['grupo2'])
    else:
        colores_estaciones.append(colores['grupo3'])

# Parámetros para la visualización
num_estaciones = len(inumet_mensual_ordenado.columns)
filas, columnas = 4, 3  # Ajusta según tu número de estaciones
fig, axes = plt.subplots(filas, columnas, figsize=(16, 12))
axes = axes.flatten()  # Convertimos la cuadrícula en un array 1D para iterar fácilmente

# Obtener los límites del eje Y
y_max = inumet_mensual_ordenado.max().max()  # Valor máximo de todas las estaciones
y_min = inumet_mensual_ordenado.min().min()  # Valor mínimo de todas las estaciones

# Graficar barras para cada estación
for i, (columna, color) in enumerate(zip(inumet_mensual_ordenado.columns, colores_estaciones)):
    axes[i].bar(inumet_mensual_ordenado.index, inumet_mensual_ordenado[columna],
                label=columna, color=color, width=20)  # Ajusta el ancho de las barras
    axes[i].set_title(f'Estación: {columna}')
    axes[i].set_xlabel('Fecha')
    axes[i].set_ylabel('Precipitación (mm)')
    axes[i].set_ylim(y_min, y_max)  # Establecer límites uniformes en Y

# Eliminar cualquier subgráfico vacío si hay más espacios que estaciones
for i in range(num_estaciones, len(axes)):
    fig.delaxes(axes[i])

# Ajustar el espaciado entre subgráficos
plt.tight_layout()


plt.savefig("inumet_mensual.jpg")

plt.show()

In [ ]:
# Boxplot INUMET Mensual con letras más grandes

# Crear la figura del boxplot
fig, ax = plt.subplots(figsize=(12, 7))
ax.boxplot(inumet_mensual, vert=True, patch_artist=True,
           boxprops=dict(facecolor='skyblue', color='black'))

# Personalización del gráfico con tamaños mayores
ax.set_title("Diagramas de caja INUMET Mensual", fontsize=18, fontweight='bold')  # antes 16
ax.set_ylabel("Precipitación (mm)", fontsize=12, fontweight='bold')  # antes 10
ax.set_xlabel("Estaciones", fontsize=11, fontweight='bold')  # antes 9
ax.set_xticklabels(inumet_mensual.columns, rotation=45, ha='right', fontsize=12, fontweight='bold')  # antes 10
ax.tick_params(axis='y', labelsize=12)  # agranda los números del eje Y
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 400)  # Escala del eje Y hasta 400 mm

plt.tight_layout()
plt.savefig("inumet_mensual_boxplot.jpg")
plt.show()

##Análisis Trimestral de las variables

In [ ]:
# Definir los períodos de las estaciones del año en 2023
periodos = [
    ("2022-12-21", "2023-03-20"),  # Verano
    ("2023-03-20", "2023-06-21"),  # Otoño
    ("2023-06-21", "2023-09-23"),  # Invierno
    ("2023-09-23", "2023-12-21"),  # Primavera
]

# Crear un DataFrame vacío para almacenar los resultados
inumet_estacional = pd.DataFrame()

# Crear un DataFrame para las fechas y las estaciones
for inicio, fin in periodos:
    # Seleccionar los datos del periodo específico y sumar la precipitación por cada columna (punto de medición)
    acumulado_periodo = inumet_diario.loc[inicio:fin].sum()  # Sumar por cada columna

    # Asignar el periodo a una nueva columna en el DataFrame
    inumet_estacional[f"{inicio} - {fin}"] = acumulado_periodo  # Guardar en el DataFrame

# Transponer el DataFrame para tener las estaciones en las filas y los periodos en las columnas
inumet_estacional = inumet_estacional.T

# Crear una nueva columna para las estaciones (Verano, Otoño, Invierno, Primavera)
inumet_estacional["estación"] = ["Verano", "Otoño", "Invierno", "Primavera"]

# Reordenar las columnas para que "estación" esté al principio
inumet_estacional = inumet_estacional[["estación"] + [col for col in inumet_estacional.columns if col != "estación"]]

# Mostrar el DataFrame resultante
inumet_estacional

In [ ]:
inumet_estacional.describe()

In [ ]:
#Gráfico de precipitación acumulada por estación del año

# Usar paleta de colores
colores = sns.color_palette("tab20", 12)  # 12 colores para cada barra

# Crear el gráfico
plt.figure(figsize=(18, 8))
bar_width = 0.8  # Ancho de cada barra
espacio_entre_estaciones = 2  # Separación entre estaciones

# Crear un eje x continuo con separación entre grupos
x_positions = []
etiquetas_estaciones = []
for i, estacion in enumerate(inumet_estacional['estación']):
    base_position = i * (12 + espacio_entre_estaciones)  # Posición base para cada estación
    x_positions.extend([base_position + j for j in range(12)])  # 12 barras por estación
    etiquetas_estaciones.append(base_position + 5.5)  # Centro del grupo de 12 barras

# Dibujar las barras
indice_color = 0
for columna in inumet_estacional.columns[1:]:
    plt.bar(x_positions[indice_color::12],  # Tomar las posiciones correspondientes a la columna
            inumet_estacional[columna],
            width=bar_width,
            label=columna,
            color=colores[indice_color])  # Asignar un color único
    indice_color += 1

# Personalizar ejes
plt.xticks(etiquetas_estaciones, inumet_estacional['estación'], rotation=0)
plt.title('Precipitación Acumulada por Estación del Año', fontsize=16)
plt.xlabel('Estación', fontsize=14)
plt.ylabel('Precipitación Acumulada (mm)', fontsize=14)

# Añadir leyenda y rejilla
plt.legend(title='Estación de Medición', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Ajustar diseño
plt.tight_layout()

# Mostrar el gráfico
plt.show()

In [ ]:
# Crear una figura con 4 subgráficos (uno para cada estación)
fig, axs = plt.subplots(2, 2, figsize=(12, 8))  # 2x2 para las 4 estaciones
axs = axs.flatten()  # Aplanar para acceder a cada subgráfico fácilmente

# Nombres de las estaciones correspondientes
nombres_estaciones = ["Verano", "Otoño", "Invierno", "Primavera"]

# Iterar sobre los períodos para crear un gráfico por estación
for i, (inicio, fin) in enumerate(periodos):
    # Filtrar los datos para cada estación
    datos_estacion = inumet.loc[inicio:fin]

    # Sumar la precipitación acumulada por cada punto de medición (columna)
    precipitaciones_acumuladas = datos_estacion.sum()

    # Calcular el promedio de precipitación acumulada
    promedio = precipitaciones_acumuladas.mean()

    # Crear el gráfico de barras para la estación correspondiente
    axs[i].bar(precipitaciones_acumuladas.index, precipitaciones_acumuladas.values, color='skyblue', edgecolor='black')

    # Agregar una línea roja para el promedio
    axs[i].axhline(y=promedio, color='red', linestyle='--', linewidth=1.5, label=f'Promedio: {promedio:.2f} mm')

    # Ajustar el título y las etiquetas del gráfico
    axs[i].set_title(f"Precipitación Acumulada: {nombres_estaciones[i]}", fontsize=14, fontweight='bold')
    axs[i].set_xlabel("Puntos de Medición", fontsize=10, fontweight='bold')
    axs[i].set_ylabel("Precipitación Acumulada (mm)", fontsize=10, fontweight='bold')

    # Ajustar las etiquetas del eje x sin usar set_xticklabels()
    axs[i].tick_params(axis='x', rotation=45, labelsize=8)  # Reducir el tamaño de las etiquetas

    # Agregar una cuadrícula para facilitar la lectura
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)

    # Establecer el límite del eje y de 0 a 600
    axs[i].set_ylim(0, 600)

    # Agregar leyenda para el promedio
    axs[i].legend(fontsize=8, loc='upper right')

# Ajustar el layout y mostrar el gráfico
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots INUMET Trimestral Sin CERO

# Seleccionar solo columnas numéricas y filtrar valores distintos de 0
inumet_estacional_numerico = inumet_estacional.drop(columns=["estación"])
datos_filtrados = [inumet_estacional_numerico[col][inumet_estacional_numerico[col] != 0].dropna()
                   for col in inumet_estacional_numerico.columns]

# Crear figura de boxplot
fig, ax = plt.subplots(figsize=(12, 7))
ax.boxplot(datos_filtrados, vert=True, patch_artist=True,
           boxprops=dict(facecolor='skyblue', color='black'))

# Personalización del gráfico
ax.set_title("Diagramas de caja INUMET Estacional (Sin 0s)", fontsize=16, fontweight='bold')
ax.set_ylabel("Precipitación (mm)", fontsize=10, fontweight='bold')
ax.set_xlabel("Estaciones", fontsize=9, fontweight='bold')
ax.set_xticklabels(inumet_estacional_numerico.columns, rotation=45, ha='right', fontsize=10, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
# Asegurarse de que el índice sea de tipo DatetimeIndex
inumet_mensual_ordenado.index = pd.to_datetime(inumet_mensual_ordenado.index)

# Resampleo trimestral: Sumamos la precipitación por trimestre
inumet_trimestral_ordenado = inumet_mensual_ordenado.resample('Q').sum()

# Limitar el rango de fechas a lo que deseas (por ejemplo, hasta el último trimestre de 2023)
inumet_trimestral_ordenado = inumet_trimestral_ordenado.loc['2022-12-21':'2023-12-21']

# Reorganiza las columnas del DataFrame según este orden
inumet_trimestral_ordenado = inumet_trimestral_ordenado[estaciones_ordenadas]

# Parámetros para la visualización
num_estaciones = len(inumet_trimestral_ordenado.columns)
filas, columnas = 4, 3  # Ajusta según tu número de estaciones
fig, axes = plt.subplots(filas, columnas, figsize=(16, 12))
axes = axes.flatten()  # Convertimos la cuadrícula en un array 1D para iterar fácilmente

# Obtener los límites del eje Y
y_max = inumet_trimestral_ordenado.max().max()  # Valor máximo de todas las estaciones
y_min = inumet_trimestral_ordenado.min().min()  # Valor mínimo de todas las estaciones

# Graficar barras para cada estación
for i, (columna, color) in enumerate(zip(inumet_trimestral_ordenado.columns, colores_estaciones)):
    axes[i].bar(inumet_trimestral_ordenado.index, inumet_trimestral_ordenado[columna],
                label=columna, color=color, width=20)  # Ajusta el ancho de las barras
    axes[i].set_title(f'Estación: {columna}')
    axes[i].set_xlabel('Fecha')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylabel('Precipitación (mm)')
    axes[i].set_ylim(y_min, y_max)  # Establecer límites uniformes en Y

# Eliminar cualquier subgráfico vacío si hay más espacios que estaciones
for i in range(num_estaciones, len(axes)):
    fig.delaxes(axes[i])

# Ajustar el espaciado entre subgráficos
plt.tight_layout()

# Mostrar el gráfico
plt.show()

##Análisis Anual de las variables

In [ ]:
# Suma todos los valores
inumet_anual = inumet_diario.sum()

inumet_anual

In [ ]:
inumet_anual.describe()

In [ ]:
# Graficar la precipitación acumulada anual por punto de medición
plt.figure(figsize=(12, 6))
plt.bar(inumet_anual.index, inumet_anual.values, color='skyblue', edgecolor='black')

# Calcular la media de la precipitación acumulada anual
media_anual = inumet_anual.values.mean()

# Agregar una línea roja para la media
plt.axhline(y=media_anual, color='red', linestyle='--', linewidth=2, label=f'Media: {media_anual:.2f} mm')

# Personalización del gráfico
plt.title("Precipitación Acumulada Anual por Punto de Medición", fontsize=16, fontweight='bold')
plt.xlabel("Puntos de Medición", fontsize=12, fontweight='bold')
plt.ylabel("Precipitación Acumulada Anual (mm)", fontsize=12, fontweight='bold')

# Ajustar etiquetas del eje x
plt.xticks(rotation=45, ha='right', fontsize=10, fontweight='bold')

# Cuadrícula
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Agregar la leyenda
plt.legend(fontsize=12, loc='upper right')

# Mostrar el gráfico
plt.tight_layout()
plt.show()

#Análisis Exploratorio CHIRPS

In [ ]:
inumet_diario.describe()

In [ ]:
chirps.describe()

In [ ]:
# 1. Eliminar la columna 'quebracho' de chirps
chirps = chirps.drop(columns='quebracho')

# 2. Recortar fechas para que coincidan con inumet_diario
fechas_comunes = chirps.index.intersection(inumet_diario.index)
chirps_recortado = chirps.loc[fechas_comunes]

In [ ]:
chirps_recortado.describe()

In [ ]:
chirps = chirps_recortado

In [ ]:
# Datos faltantes
nans_chirps = null_report(chirps)
print(nans_chirps)

##Análisis Diario de las variables

In [ ]:
# Histogramas
fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

min_val = chirps.min().min()
max_val = chirps.max().max()

max_freq = 0
for columna in chirps.columns:
    datos_filtrados = chirps[columna].dropna()
    _, bins = np.histogram(datos_filtrados, bins=30)
    hist, _ = np.histogram(datos_filtrados, bins=bins)
    max_freq = max(max_freq, hist.max())

for i, columna in enumerate(chirps.columns):
    datos_filtrados = chirps[columna].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].set_xlabel('Precipitación (mm)', fontsize=9, fontweight='bold')
    axs[i].set_ylabel('Frecuencia', fontsize=9, fontweight='bold')
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(min_val, max_val)  # Limitar el eje x
    axs[i].set_ylim(0, max_freq)       # Limitar el eje y

fig.suptitle("Histogramas CHIRPS Datos Diarios", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Histogramas CHIRPS sin valores 0 (con ejes generales y límites fijos)
fig, axs = plt.subplots(4, 3, figsize=(12, 10))
axs = axs.flatten()

# Límites fijos
x_min, x_max = 0, 125
y_min, y_max = 0, 60

# Graficar cada histograma
for i, columna in enumerate(chirps.columns):
    datos_filtrados = chirps[columna][chirps[columna] != 0].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)

    #Ejes unificados y fijos
    axs[i].set_xlim(x_min, x_max)
    axs[i].set_ylim(y_min, y_max)

    # Ajustar tamaño de ticks
    axs[i].tick_params(axis='both', which='major', labelsize=10)

# Eliminar subplots sobrantes
if len(chirps.columns) < len(axs):
    for j in range(len(chirps.columns), len(axs)):
        fig.delaxes(axs[j])

# Etiquetas generales de los ejes
fig.supxlabel("Precipitación (mm)", fontsize=14, fontweight='bold')
fig.supylabel("Frecuencia", fontsize=14, fontweight='bold')

# Título general
fig.suptitle("Histogramas CHIRPS Datos Diarios (Sin 0s)", fontsize=16, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("chirps_hist_diariosin0.jpg", dpi=300)
plt.show()

In [ ]:
# Histogramas CHIRPS sin valores 0 (con ejes generales y límites fijos)
fig, axs = plt.subplots(4, 3, figsize=(10, 8))
axs = axs.flatten()

# Límites fijos
x_min, x_max = 0, 125
y_min, y_max = 0, 60

# Graficar cada histograma
for i, columna in enumerate(chirps.columns):
    datos_filtrados = chirps[columna][chirps[columna] != 0].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)

    # Ejes unificados y fijos
    axs[i].set_xlim(x_min, x_max)
    axs[i].set_ylim(y_min, y_max)

    #Ajustar tamaño de ticks (ahora 12, igual al segundo gráfico)
    axs[i].tick_params(axis='both', which='major', labelsize=12)

# Eliminar subplots sobrantes
if len(chirps.columns) < len(axs):
    for j in range(len(chirps.columns), len(axs)):
        fig.delaxes(axs[j])

# Etiquetas generales de los ejes
fig.supxlabel("Precipitación (mm)", fontsize=14, fontweight='bold')
fig.supylabel("Frecuencia", fontsize=14, fontweight='bold')

# Título general
fig.suptitle("Histogramas CHIRPS Datos Diarios (Sin 0s)", fontsize=16, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("chirps_hist_diariosin0.jpg", dpi=300)
plt.show()

In [ ]:
# Minimos por estación en datos diarios
minimos = chirps.min()
print(minimos)

In [ ]:
# Minimos distintos de 0 por estación en datos diarios
minimos_sin_cero = chirps[chirps != 0].min()
print(minimos_sin_cero)

In [ ]:
# Maximos por estación en datos diarios
maximos = chirps.max()
print(maximos)

In [ ]:
import matplotlib.dates as mdates

# Series Temporales CHIRPS Diario

# Configuración de la cuadrícula
fig, axs = plt.subplots(4, 3, figsize=(12, 10))
axs = axs.flatten()

# Definir el límite máximo de fecha
inicio1 = chirps.index.min()
fin1 = pd.Timestamp('2023-12-21')

# Crear un gráfico de líneas para cada estación
for i, columna in enumerate(chirps.columns):
    axs[i].plot(chirps.index, chirps[columna], label=columna, color='steelblue')
    axs[i].set_title(f'{columna}', fontsize=14)  # 🔹 Igualado al gráfico INUMET
    axs[i].set_ylim(0, 125)  # Escala fija del eje Y
    axs[i].set_xlim(inicio1, fin1)
    axs[i].grid(True, axis='y', linestyle='--', alpha=0.7)

    # Ticks homogéneos
    axs[i].tick_params(axis='both', which='major', labelsize=12)

    # Fechas en formato mm-yy
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%m-%y'))

# Eliminar subplots vacíos si hay menos estaciones que paneles
if len(chirps.columns) < len(axs):
    for j in range(len(chirps.columns), len(axs)):
        fig.delaxes(axs[j])

# Etiquetas generales de los ejes
fig.supxlabel("Fecha", fontsize=14, fontweight='bold')
fig.supylabel("Precipitación (mm)", fontsize=14, fontweight='bold')

# Título general
fig.suptitle("Series Temporales CHIRPS diario", fontsize=16, fontweight='bold')

# Ajustar el diseño
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar figura
plt.savefig("series_temporales_chirps_diario.png", dpi=300)

# Mostrar figura
plt.show()

In [ ]:
# Boxplots
# Creamos la figura de un solo boxplot para todas las columnas
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(chirps, vert=True, patch_artist=True, boxprops=dict(facecolor='skyblue', color='black'))

# Ajustamos el título y las etiquetas de los ejes
ax.set_title("Diagramas de caja CHIRPS Diario", fontsize=16, fontweight='bold')
ax.set_ylabel("Precipitación (mm)", fontsize=9, fontweight='bold')
ax.set_xlabel("Estaciones", fontsize=9, fontweight='bold')
ax.set_xticklabels(chirps.columns, rotation=45, ha='right', fontsize=9, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Mostramos el gráfico
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots

# Filtramos valores diferentes de 0 y los almacenamos en una lista
datos_filtrados = [chirps[columna][chirps[columna] != 0].dropna() for columna in chirps.columns]

# Creamos la figura de un solo boxplot
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(datos_filtrados, vert=True, patch_artist=True, boxprops=dict(facecolor='skyblue', color='black'))

# Ajustamos el título y las etiquetas de los ejes
ax.set_title("Diagramas de caja CHIRPS Diario (Sin 0s)", fontsize=16, fontweight='bold')
ax.set_ylabel("Precipitación (mm)", fontsize=9, fontweight='bold')
ax.set_xlabel("Estaciones", fontsize=9, fontweight='bold')
ax.set_xticklabels(chirps.columns, rotation=45, ha='right', fontsize=9, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Mostramos el gráfico
plt.tight_layout()
plt.show()

In [ ]:
chirps = chirps.rename(columns={
    "paradorlavibora": "lavibora",
    "elaguila": "aguila"
})

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

# Configuración de figura
fig, axs = plt.subplots(4, 3, figsize=(16, 12))
axs = axs.flatten()

# Escalas fijas para todos los subgráficos
xlim = (0, 125)
ylim = (0, 125)

for i, columna in enumerate(inumet_diario.columns):
    if columna in chirps.columns:
        # Eliminar filas con NaN en ambas fuentes
        x = inumet_diario[columna]   # INUMET en X
        y = chirps[columna]          # CHIRPS en Y
        valid = x.notna() & y.notna()
        x = x[valid]
        y = y[valid]

        # Datos para regresión
        X = x.values.reshape(-1, 1)
        Y = y.values

        # Regresión lineal
        model = LinearRegression()
        model.fit(X, Y)
        y_pred = model.predict(X)
        r2 = model.score(X, Y)
        corr, _ = pearsonr(x, y)

        # Plot
        axs[i].scatter(x, y, alpha=0.5, label="Datos", color='steelblue')
        axs[i].plot(x, y_pred, color='red', linewidth=2, label="Línea de Tendencia")
        axs[i].set_title(f'{columna}', fontsize=16)  # 🔹 antes 14
        axs[i].set_xlim(xlim)
        axs[i].set_ylim(ylim)
        axs[i].grid(True, linestyle='--', alpha=0.7)

        # Texto con R² y correlación dentro de la gráfica
        axs[i].text(0.05, 0.95, f"$R^2$ = {r2:.2f}\nCorrelación = {corr:.2f}",
                    fontsize=14, style='italic', transform=axs[i].transAxes,  # 🔹 antes 12
                    verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

        # Ticks más grandes
        axs[i].tick_params(axis='both', which='major', labelsize=14)  # 🔹 antes 12

# Eliminar subplots vacíos
if len(inumet_diario.columns) < len(axs):
    for j in range(len(inumet_diario.columns), len(axs)):
        fig.delaxes(axs[j])

# Etiquetas generales de los ejes
fig.supxlabel("INUMET (mm)", fontsize=16, fontweight='bold')  # 🔹 antes 14
fig.supylabel("CHIRPS (mm)", fontsize=16, fontweight='bold')  # 🔹 antes 14

# Título general
fig.suptitle("Dispersión Diaria INUMET - CHIRPS", fontsize=18, fontweight='bold')  # 🔹 antes 16

# Ajustar layout
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar y mostrar
plt.savefig("dispersión_chirps_vs_inumet.png", dpi=300)
plt.show()

##Análisis Mensual de las variables

In [ ]:
# Definir los períodos mensuales del 21 de un mes al 21 del siguiente
periodos = [
    ("2022-12-21", "2023-01-21"),
    ("2023-01-21", "2023-02-21"),
    ("2023-02-21", "2023-03-21"),
    ("2023-03-21", "2023-04-21"),
    ("2023-04-21", "2023-05-21"),
    ("2023-05-21", "2023-06-21"),
    ("2023-06-21", "2023-07-21"),
    ("2023-07-21", "2023-08-21"),
    ("2023-08-21", "2023-09-21"),
    ("2023-09-21", "2023-10-21"),
    ("2023-10-21", "2023-11-21"),
    ("2023-11-21", "2023-12-21")
]

# Crear un DataFrame vacío para almacenar los resultados
chirps_mensual = pd.DataFrame()

# Calcular la precipitación acumulada para cada período
for inicio, fin in periodos:
    # Seleccionar los datos dentro del intervalo específico y sumar las precipitaciones por columna
    acumulado_periodo = chirps.loc[inicio:fin].sum()  # Sumar por cada columna (punto de medición)

    # Agregar los datos acumulados al DataFrame de resultados
    chirps_mensual[f"{inicio} - {fin}"] = acumulado_periodo

# Transponer el DataFrame para tener los períodos como filas
chirps_mensual = chirps_mensual.T

# Crear una columna con las fechas iniciales del período
chirps_mensual["fecha_inicio"] = [inicio for inicio, fin in periodos]

# Reorganizar las columnas para que "fecha_inicio" esté al principio
chirps_mensual = chirps_mensual[["fecha_inicio"] + [col for col in chirps_mensual.columns if col != "fecha_inicio"]]

# Convertir la columna 'fecha_inicio' a datetime
chirps_mensual['fecha_inicio'] = pd.to_datetime(chirps_mensual['fecha_inicio'], format='%Y-%m-%d')

# Configurar 'fecha_inicio' como índice del DataFrame
chirps_mensual.set_index('fecha_inicio', inplace=True)

# Ordenar el DataFrame por el índice temporal
chirps_mensual.sort_index(inplace=True)

# Mostrar el DataFrame resultante
chirps_mensual

In [ ]:
chirps_mensual.describe()

In [ ]:
# Datos faltantes
nans_chirps_mensual = null_report(chirps_mensual)
print(nans_chirps_mensual)

In [ ]:
# Histogramas

fig, axs = plt.subplots(4, 3, figsize=(12.25, 10))
axs = axs.flatten()

min_val = chirps_mensual.min().min()
max_val = chirps_mensual.max().max()

max_freq = 0
for columna in chirps_mensual.columns:
    datos_filtrados = chirps_mensual[columna].dropna()
    _, bins = np.histogram(datos_filtrados, bins=30)
    hist, _ = np.histogram(datos_filtrados, bins=bins)
    max_freq = max(max_freq, hist.max())

for i, columna in enumerate(chirps_mensual.columns):
    datos_filtrados = chirps_mensual[columna].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].set_xlabel('Precipitación (mm)', fontsize=9, fontweight='bold')
    axs[i].set_ylabel('Frecuencia', fontsize=9, fontweight='bold')
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(min_val, max_val)  # Limitar el eje x
    axs[i].set_ylim(0, max_freq)       # Limitar el eje y

fig.suptitle("Histogramas CHIRPS Datos Mensuales", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Histogramas sin valores 0

fig, axs = plt.subplots(4, 3, figsize=(12.25, 10))
axs = axs.flatten()

min_val = chirps_mensual[chirps_mensual != 0].min().min()
max_val = chirps_mensual[chirps_mensual != 0].max().max()

max_freq = 0
for columna in chirps.columns:
    datos_filtrados = chirps_mensual[columna][chirps_mensual[columna] != 0].dropna()
    hist, _ = np.histogram(datos_filtrados, bins=30)
    max_freq = max(max_freq, hist.max())

for i, columna in enumerate(chirps_mensual.columns):
    datos_filtrados = chirps_mensual[columna][chirps_mensual[columna] != 0].dropna()
    axs[i].hist(datos_filtrados, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{columna}', fontsize=14)
    axs[i].set_xlabel('Precipitación (mm)', fontsize=9, fontweight='bold')
    axs[i].set_ylabel('Frecuencia', fontsize=9, fontweight='bold')
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].grid(axis='x', linestyle='--', alpha=0.7)
    axs[i].set_xlim(min_val, max_val)  # Limitar el eje x
    axs[i].set_ylim(0, max_freq)       # Limitar el eje y

fig.suptitle("Histogramas CHIRPS Datos Mensuales (Sin 0s)", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Minimos por estación en datos mensuales
minimos_m = chirps_mensual.min()
print(minimos_m)

In [ ]:
# Serie temporal lineas datos mensuales

plt.figure(figsize=(10, 8))

# Graficar cada columna
for columna in chirps_mensual.columns:
    plt.plot(chirps_mensual.index, chirps_mensual[columna], label=columna)

# Personalización del gráfico
plt.title("Serie Temporal Datos Mensuales CHIRPS", fontsize=16, fontweight='bold')
plt.xlabel("Tiempo", fontsize=12, fontweight='bold')
plt.ylabel("Precipitación Acumulada (mm)", fontsize=12, fontweight='bold')

# Formato del eje x
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%m'))

# Cuadrícula mensual
plt.grid(axis='x', which='major', linestyle='--', alpha=0.7)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Leyenda
plt.legend(title="Columnas", fontsize=10)

# Ajustar diseño
plt.tight_layout()

# Mostrar el gráfico
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Renombrar columnas en CHIRPS para que coincidan con INUMET
renombres = {
    "paradorlavibora": "lavibora",
    "elaguila": "aguila"
}
chirps_mensual = chirps_mensual.rename(columns=renombres)

# Crear figura con dos subplots que comparten eje X
fig, axs = plt.subplots(2, 1, figsize=(14, 11), sharex=True)

columnas = inumet_mensual.columns

# Gráfico superior: INUMET
for columna in columnas:
    axs[0].plot(inumet_mensual.index, inumet_mensual[columna], label=columna)

axs[0].set_title("Serie Temporal Datos Mensuales INUMET", fontsize=20, fontweight='bold')
axs[0].set_ylabel("Precipitación Acumulada (mm)", fontsize=16, fontweight='bold')
axs[0].set_ylim(0, 400)
axs[0].tick_params(axis='both', labelsize=14)
axs[0].grid(axis='x', which='major', linestyle='--', alpha=0.7)
axs[0].grid(axis='y', linestyle='--', alpha=0.7)

# Gráfico inferior: CHIRPS
for columna in columnas:
    if columna in chirps_mensual.columns:
        axs[1].plot(chirps_mensual.index, chirps_mensual[columna])

axs[1].set_title("Serie Temporal Datos Mensuales CHIRPS", fontsize=20, fontweight='bold')
axs[1].set_xlabel("Meses", fontsize=16, fontweight='bold')
axs[1].set_ylabel("Precipitación Acumulada (mm)", fontsize=16, fontweight='bold')
axs[1].set_ylim(0, 400)
axs[1].tick_params(axis='both', labelsize=14)
axs[1].grid(axis='x', which='major', linestyle='--', alpha=0.7)
axs[1].grid(axis='y', linestyle='--', alpha=0.7)

# Formato de eje x con mes y año
axs[1].xaxis.set_major_locator(mdates.MonthLocator())
axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%y'))

# Leyenda global debajo, centrada y horizontal con fuentes más grandes
fig.legend(columnas, title="Estaciones", fontsize=15, title_fontsize=17,
           loc='upper center', bbox_to_anchor=(0.5, -0.07), ncol=4, frameon=False)

# Ajustar layout para dejar más espacio para la leyenda
plt.tight_layout(rect=[0, 0, 1, 0.92])

# Guardar y mostrar
plt.savefig("comparacion_series_temporales_mensuales.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Configuración de la cuadrícula
fig, axs = plt.subplots(4, 3, figsize=(16, 12))
axs = axs.flatten()

# Obtener los límites del eje Y para todas las estaciones
y_min = chirps_mensual.min().min()  # Valor mínimo de todas las estaciones
y_max = chirps_mensual.max().max()  # Valor máximo de todas las estaciones

# Define el límite máximo de fecha
inicio1 = chirps_mensual.index.min()
fin1 = pd.Timestamp('2023-12-21')  # Fecha máxima a mostrar

# Crear un gráfico de barras para cada estación
for i, columna in enumerate(chirps_mensual.columns):
    axs[i].bar(chirps_mensual.index, chirps_mensual[columna], label=columna, color='steelblue', width=1.0, edgecolor='red')
    axs[i].set_title(f'Estación: {columna}', fontsize=14)
    axs[i].set_xlabel('Fecha', fontsize=9, fontweight='bold')
    axs[i].set_ylabel('Precipitación (mm)', fontsize=9, fontweight='bold')
    axs[i].set_ylim(y_min, y_max)  # Establecer la misma escala en el eje Y
    axs[i].set_xlim(inicio1, fin1)  # Fija los límites del eje X
    axs[i].grid(True, axis='y', linestyle='--', alpha=0.7)

# Título general del gráfico
fig.suptitle("Series Temporales CHIRPS mensual", fontsize=16, fontweight='bold')

# Ajusta el espaciado entre subgráficos
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Boxplots CHIRPS Mensual con letras más grandes

# Creamos la figura de un solo boxplot para todas las columnas
fig, ax = plt.subplots(figsize=(12, 7))
ax.boxplot(chirps_mensual, vert=True, patch_artist=True,
           boxprops=dict(facecolor='skyblue', color='black'))

# Ajustamos el título y las etiquetas de los ejes con mayor tamaño
ax.set_title("Diagramas de caja CHIRPS Mensual", fontsize=18, fontweight='bold')  # antes 16
ax.set_ylabel("Precipitación (mm)", fontsize=11, fontweight='bold')  # antes 9
ax.set_xlabel("Estaciones", fontsize=11, fontweight='bold')  # antes 9
ax.set_xticklabels(chirps.columns, rotation=45, ha='right',
                   fontsize=11, fontweight='bold')  # antes 9
ax.tick_params(axis='y', labelsize=11)  # agranda los números del eje Y
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 400)  # Escala del eje Y hasta 400 mm

# Mostramos el gráfico
plt.tight_layout()
plt.savefig("chirps_mensual_boxplot.jpg")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

# Renombrar columnas en CHIRPS para que coincidan con INUMET
renombres = {
    "paradorlavibora": "lavibora",
    "elaguila": "aguila"
}
chirps_mensual = chirps_mensual.rename(columns=renombres)

# Configuración de figura
fig, axs = plt.subplots(4, 3, figsize=(16, 12))
axs = axs.flatten()

# Escalas fijas para todos los subgráficos
xlim = (0, 400)
ylim = (0, 400)

for i, columna in enumerate(inumet_mensual.columns):
    if columna in chirps_mensual.columns:
        # Eliminar filas con NaN en ambas fuentes
        x = inumet_mensual[columna]
        y = chirps_mensual[columna]
        valid = x.notna() & y.notna()
        x = x[valid]
        y = y[valid]

        # Regresión lineal
        X = x.values.reshape(-1, 1)
        Y = y.values
        model = LinearRegression()
        model.fit(X, Y)
        y_pred = model.predict(X)
        r2 = model.score(X, Y)
        corr, _ = pearsonr(x, y)

        # Plot
        axs[i].scatter(x, y, alpha=0.5, color='steelblue')
        axs[i].plot(x, y_pred, color='red', linewidth=2)
        axs[i].set_title(f'{columna}', fontsize=18)
        axs[i].set_xlim(xlim)
        axs[i].set_ylim(ylim)
        axs[i].grid(True, linestyle='--', alpha=0.7)
        axs[i].tick_params(axis='both', labelsize=14)

        # Texto con R² y correlación dentro de la gráfica
        axs[i].text(0.05, 0.95, f"$R^2$ = {r2:.2f}\nCorr = {corr:.2f}",
                    fontsize=12, style='italic', transform=axs[i].transAxes,
                    verticalalignment='top', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

# Eliminar subplots vacíos
if len(inumet_mensual.columns) < len(axs):
    for j in range(len(inumet_mensual.columns), len(axs)):
        fig.delaxes(axs[j])

# Etiquetas generales de ejes
fig.supxlabel("Precipitación INUMET (mm)", fontsize=16, fontweight='bold')
fig.supylabel("Precipitación CHIRPS (mm)", fontsize=16, fontweight='bold')

# Título general
fig.suptitle("Dispersión Mensual INUMET - CHIRPS", fontsize=20, fontweight='bold')

# Ajuste de layout
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar y mostrar
plt.savefig("dispersión_chirps_vs_inumet_mensual.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

# Renombrar columnas en CHIRPS para que coincidan con INUMET
renombres = {
    "paradorlavibora": "lavibora",
    "elaguila": "aguila"
}
chirps_mensual = chirps_mensual.rename(columns=renombres)

# Configuración de figura
fig, axs = plt.subplots(4, 3, figsize=(16, 12))
axs = axs.flatten()

# Escalas fijas para todos los subgráficos
xlim = (0, 400)
ylim = (0, 400)

# Tamaños de letra ajustados
title_size = 20  # títulos de subplots
label_size = 18  # ejes
tick_size = 16   # ticks
text_size = 14   # R² y corr

for i, columna in enumerate(inumet_mensual.columns):
    if columna in chirps_mensual.columns:
        # Eliminar filas con NaN en ambas fuentes
        x = inumet_mensual[columna]
        y = chirps_mensual[columna]
        valid = x.notna() & y.notna()
        x = x[valid]
        y = y[valid]

        # Regresión lineal
        X = x.values.reshape(-1, 1)
        Y = y.values
        model = LinearRegression()
        model.fit(X, Y)
        y_pred = model.predict(X)
        r2 = model.score(X, Y)
        corr, _ = pearsonr(x, y)

        # Plot
        axs[i].scatter(x, y, alpha=0.5, color='steelblue')
        axs[i].plot(x, y_pred, color='red', linewidth=2)
        axs[i].set_title(f'{columna}', fontsize=title_size)
        axs[i].set_xlim(xlim)
        axs[i].set_ylim(ylim)
        axs[i].grid(True, linestyle='--', alpha=0.7)
        axs[i].tick_params(axis='both', labelsize=tick_size)

        # Texto con R² y correlación dentro de la gráfica
        axs[i].text(0.05, 0.95, f"$R^2$ = {r2:.2f}\nCorr = {corr:.2f}",
                    fontsize=text_size, style='italic', transform=axs[i].transAxes,
                    verticalalignment='top', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

# Eliminar subplots vacíos
if len(inumet_mensual.columns) < len(axs):
    for j in range(len(inumet_mensual.columns), len(axs)):
        fig.delaxes(axs[j])

# Etiquetas generales de ejes
fig.supxlabel("Precipitación INUMET (mm)", fontsize=label_size, fontweight='bold')
fig.supylabel("Precipitación CHIRPS (mm)", fontsize=label_size, fontweight='bold')

# Título general
fig.suptitle("Dispersión Mensual INUMET - CHIRPS", fontsize=22, fontweight='bold')

# Ajuste de layout
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar y mostrar
plt.savefig("dispersión_chirps_vs_inumet_mensual_grande.png", dpi=300)
plt.show()

##Análisis Trimestral de las variables

In [ ]:
# Definir los períodos de las estaciones del año en 2023
periodos = [
    ("2022-12-21", "2023-03-20"),  # Verano
    ("2023-03-20", "2023-06-21"),  # Otoño
    ("2023-06-21", "2023-09-23"),  # Invierno
    ("2023-09-23", "2023-12-21"),  # Primavera
]

# Crear un DataFrame vacío para almacenar los resultados
chirps_estacional = pd.DataFrame()

# Crear un DataFrame para las fechas y las estaciones
for inicio, fin in periodos:
    # Seleccionar los datos del periodo específico y sumar la precipitación por cada columna (punto de medición)
    acumulado_periodo = chirps.loc[inicio:fin].sum()  # Sumar por cada columna

    # Asignar el periodo a una nueva columna en el DataFrame
    chirps_estacional[f"{inicio} - {fin}"] = acumulado_periodo  # Guardar en el DataFrame

# Transponer el DataFrame para tener las estaciones en las filas y los periodos en las columnas
chirps_estacional = chirps_estacional.T

# Crear una nueva columna para las estaciones (Verano, Otoño, Invierno, Primavera)
chirps_estacional["estación"] = ["Verano", "Otoño", "Invierno", "Primavera"]

# Reordenar las columnas para que "estación" esté al principio
chirps_estacional = chirps_estacional[["estación"] + [col for col in chirps_estacional.columns if col != "estación"]]

# Mostrar el DataFrame resultante
chirps_estacional

In [ ]:
chirps_estacional.describe()

In [ ]:
# Boxplots

# Seleccionamos solo las columnas numéricas para el boxplot
chirps_estacional_numerico = chirps_estacional.drop(columns=["estación"])

# Creamos la figura de un solo boxplot para todas las columnas
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(chirps_estacional_numerico, vert=True, patch_artist=True, boxprops=dict(facecolor='skyblue', color='black'))

# Ajustamos el título y las etiquetas de los ejes
ax.set_title("Diagramas de caja CHIRPS Estacional", fontsize=16, fontweight='bold')
ax.set_ylabel("Precipitación (mm)", fontsize=9, fontweight='bold')
ax.set_xlabel("Estaciones", fontsize=9, fontweight='bold')
ax.set_xticklabels(chirps_estacional_numerico.columns, rotation=45, ha='right', fontsize=9, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Mostramos el gráfico
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Crear figura con 4 subplots (2x2)
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
axs = axs.flatten()

# Iterar por cada fila (estación solar)
for i in range(4):
    estacion_nombre = chirps_estacional['estación'].iloc[i]
    precipitaciones = chirps_estacional.drop(columns=['estación']).iloc[i]
    promedio = precipitaciones.mean()

    axs[i].bar(precipitaciones.index, precipitaciones.values, color='skyblue', edgecolor='black')
    axs[i].axhline(y=promedio, color='red', linestyle='--', linewidth=1.5,
                   label=f'Promedio estaciones: {promedio:.2f} mm')

    axs[i].set_title(f"Precipitación Acumulada: {estacion_nombre}", fontsize=14, fontweight='bold')
    axs[i].set_xlabel("Estaciones Meteorológicas", fontsize=10, fontweight='bold')
    axs[i].set_ylabel("Precipitación (mm)", fontsize=10, fontweight='bold')
    axs[i].tick_params(axis='x', rotation=45, labelsize=8)
    axs[i].set_ylim(0, 650)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].legend(fontsize=8, loc='upper right')

# Ajuste final
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filtramos columnas comunes
columnas_comunes = chirps_estacional.columns.intersection(inumet_estacional.columns)
columnas_comunes = columnas_comunes.drop('estación') if 'estación' in columnas_comunes else columnas_comunes

chirps_filtrado = chirps_estacional[columnas_comunes].copy()
inumet_filtrado = inumet_estacional[columnas_comunes].copy()

# Crear figura
fig, axs = plt.subplots(2, 2, figsize=(13, 9))
axs = axs.flatten()

# Estaciones solares fijas en orden
nombres_estaciones = ["Verano", "Otoño", "Invierno", "Primavera"]

x = np.arange(len(columnas_comunes))
width = 0.4

# Tamaños de fuente unificados
title_size = 18
label_size = 15
tick_size = 13
legend_size = 12

# Gráficos
for i in range(4):
    chirps_vals = chirps_filtrado.iloc[i]
    inumet_vals = inumet_filtrado.iloc[i]

    prom_chirps = chirps_vals.mean()
    prom_inumet = inumet_vals.mean()

    # Barras
    axs[i].bar(x - width/2, chirps_vals.values, width=width, color='orange', edgecolor='black', label='CHIRPS')
    axs[i].bar(x + width/2, inumet_vals.values, width=width, color='skyblue', edgecolor='black', label='INUMET')

    # Líneas de promedio
    axs[i].axhline(y=prom_chirps, color='darkorange', linestyle='--', linewidth=1.5, label=f'Prom. CHIRPS: {prom_chirps:.1f}')
    axs[i].axhline(y=prom_inumet, color='deepskyblue', linestyle='--', linewidth=1.5, label=f'Prom. INUMET: {prom_inumet:.1f}')

    axs[i].set_title(f"{nombres_estaciones[i]}", fontsize=title_size, fontweight='bold')
    axs[i].set_xticks(x)
    axs[i].set_xticklabels(columnas_comunes, rotation=45, ha='right', fontsize=tick_size)
    axs[i].set_ylim(0, 650)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    axs[i].legend(fontsize=legend_size, loc='upper right')

# Títulos unificados de ejes
fig.text(0.5, 0.04, "Estaciones Meteorológicas", ha='center', fontsize=label_size, fontweight='bold')
fig.text(0.04, 0.5, "Precipitación Acumulada (mm)", va='center', rotation='vertical', fontsize=label_size, fontweight='bold')

plt.tight_layout(rect=[0.05, 0.05, 1, 0.96])
plt.savefig("comparacion_estacional_chirps_inumet_unificado.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Asegurar columnas comunes y ordenadas
columnas_comunes = chirps_estacional.columns.intersection(inumet_estacional.columns)
columnas_comunes = columnas_comunes.drop('estación') if 'estación' in columnas_comunes else columnas_comunes

chirps_filtrado = chirps_estacional[columnas_comunes].copy()
inumet_filtrado = inumet_estacional[columnas_comunes].copy()

# Asignar manualmente los nombres de las estaciones solares
estaciones_solares = ["Verano", "Otoño", "Invierno", "Primavera"]

# Colores para cada estación
colores = ['gold', 'orange', 'skyblue', 'limegreen']
x = np.arange(len(columnas_comunes))
width = 0.2

# Tamaños de fuente
title_size = 18
label_size = 15
tick_size = 13
legend_size = 13

fig, axs = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

#Subplot superior: INUMET
bars_handles = []
lines_handles = []

for i in range(4):
    vals = inumet_filtrado.iloc[i].values
    offset = (i - 1.5) * width  # Ajuste para 4 estaciones

    # Barras
    bar = axs[0].bar(x + offset, vals, width=width, color=colores[i], edgecolor='black')
    bars_handles.append(bar[0])

    # Línea promedio
    promedio = vals.mean()
    line = axs[0].axhline(y=promedio, color=colores[i], linestyle='--', linewidth=1.5)
    lines_handles.append(line)

axs[0].set_title("Precipitación Estacional por Estación Meteorológica - INUMET",
                 fontsize=title_size, fontweight='bold')
axs[0].set_ylabel("Precipitación (mm)", fontsize=label_size, fontweight='bold')
axs[0].set_ylim(0, 650)
axs[0].grid(axis='y', linestyle='--', alpha=0.7)
axs[0].tick_params(axis='both', labelsize=tick_size)

# Leyendas
leg_barras_0 = axs[0].legend(bars_handles, estaciones_solares, title="Estación Solar",
                             fontsize=legend_size, title_fontsize=legend_size, loc='upper left')
labels_promedios_0 = [f'{est} {vals.mean():.1f} mm' for est, vals in zip(estaciones_solares, inumet_filtrado.values)]
leg_lineas_0 = axs[0].legend(lines_handles, labels_promedios_0, title="Promedios",
                             fontsize=legend_size, title_fontsize=legend_size, loc='upper right')
axs[0].add_artist(leg_barras_0)

# Subplot inferior: CHIRPS
lines_handles_1 = []

for i in range(4):
    vals = chirps_filtrado.iloc[i].values
    offset = (i - 1.5) * width

    # Barras
    axs[1].bar(x + offset, vals, width=width, color=colores[i], edgecolor='black')

    # Línea promedio
    promedio = vals.mean()
    line = axs[1].axhline(y=promedio, color=colores[i], linestyle='--', linewidth=1.5)
    lines_handles_1.append(line)

axs[1].set_title("Precipitación Estacional por Estación Meteorológica - CHIRPS",
                 fontsize=title_size, fontweight='bold')
axs[1].set_xlabel("Estaciones Meteorológicas", fontsize=label_size, fontweight='bold')
axs[1].set_ylabel("Precipitación (mm)", fontsize=label_size, fontweight='bold')
axs[1].set_xticks(x)
axs[1].set_xticklabels(columnas_comunes, rotation=45, ha='right', fontsize=tick_size)
axs[1].set_ylim(0, 650)
axs[1].grid(axis='y', linestyle='--', alpha=0.7)
axs[1].tick_params(axis='both', labelsize=tick_size)

# Leyenda líneas (promedios)
labels_promedios_1 = [f'{est} {vals.mean():.1f} mm' for est, vals in zip(estaciones_solares, chirps_filtrado.values)]
axs[1].legend(lines_handles_1, labels_promedios_1, title="Promedios",
              fontsize=legend_size, title_fontsize=legend_size, loc='upper right')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("comparacion_estacional_chirps_inumet_estacionesfijas.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

# Configuración de figura
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
axs = axs.flatten()

# Nombres de las estaciones solares
nombres_estaciones_solares = ['Verano', 'Otoño', 'Invierno', 'Primavera']

# Eliminar columna "estación" si existe
inumet_vals = inumet_estacional.drop(columns='estación') if 'estación' in inumet_estacional.columns else inumet_estacional.copy()
chirps_vals = chirps_estacional.drop(columns='estación') if 'estación' in chirps_estacional.columns else chirps_estacional.copy()

# Estaciones meteorológicas y colores
estaciones_meteorologicas = inumet_vals.columns
colores = plt.cm.tab20.colors
colores_dict = {est: colores[i % len(colores)] for i, est in enumerate(estaciones_meteorologicas)}

# Crear handles para la leyenda
handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=colores_dict[est],
               markeredgecolor='black',
               markersize=10, label=est)
    for est in estaciones_meteorologicas
]

# Tamaños de fuente
title_size = 18
label_size = 15
tick_size = 13
legend_size = 13
r2_size = 12

# Iterar sobre cada estación solar
for i in range(4):
    x = inumet_vals.iloc[i]
    y = chirps_vals.iloc[i]

    # Eliminar valores NaN
    valid = x.notna() & y.notna()
    x = x[valid]
    y = y[valid]
    etiquetas = estaciones_meteorologicas[valid]

    # Datos para regresión
    X = x.values.reshape(-1, 1)
    Y = y.values
    model = LinearRegression()
    model.fit(X, Y)
    y_pred = model.predict(X)
    r2 = model.score(X, Y)
    corr, _ = pearsonr(x, y)

    # Dibujar puntos con color por estación
    for xi, yi, etiqueta in zip(x, y, etiquetas):
        axs[i].scatter(xi, yi, color=colores_dict[etiqueta], s=80, edgecolor='black', linewidth=0.5)
        # Desplazar etiquetas ligeramente hacia arriba
        axs[i].text(xi + 0.01*xi, yi + 0.05*yi, etiqueta, fontsize=tick_size, color=colores_dict[etiqueta],
                    weight='bold', alpha=0.9)

    # Línea de regresión
    axs[i].plot(x, y_pred, color='red', linewidth=2)

    # Título de cada subplot con estación solar
    axs[i].set_title(nombres_estaciones_solares[i], fontsize=title_size, fontweight='bold')
    axs[i].grid(True, linestyle='--', alpha=0.7)
    axs[i].tick_params(axis='both', labelsize=tick_size)

    # Escalas dinámicas por plot
    margen = 30
    axs[i].set_xlim(x.min() - margen, x.max() + margen)
    axs[i].set_ylim(y.min() - margen, y.max() + margen)

    # Texto con R² y correlación
    axs[i].text(0.05, 0.95, f"$R^2$ = {r2:.2f}\nCorrelación = {corr:.2f}",
                fontsize=r2_size, style='italic', transform=axs[i].transAxes,
                verticalalignment='top', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

# Leyenda debajo del eje centrada, en dos filas
fig.legend(handles=handles, title="Estaciones Meteorológicas",
           loc='upper center', bbox_to_anchor=(0.5, -0.12),
           ncol=int(np.ceil(len(estaciones_meteorologicas)/2)), fontsize=legend_size, title_fontsize=legend_size)

# Títulos unificados de ejes
fig.text(0.5, 0.04, "INUMET (mm)", ha='center', fontsize=label_size, fontweight='bold')
fig.text(0.04, 0.5, "CHIRPS (mm)", va='center', rotation='vertical', fontsize=label_size, fontweight='bold')

# Título general
fig.suptitle("Dispersión Estacional INUMET - CHIRPS", fontsize=title_size, fontweight='bold')

plt.tight_layout(rect=[0.05, 0.1, 0.95, 0.96])
plt.savefig("dispersión_chirps_vs_inumet_estacional_leyenda_dos_filas.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
pip install adjustText

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Configuración de figura
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
axs = axs.flatten()

# Nombres de las estaciones solares
nombres_estaciones_solares = ['Verano', 'Otoño', 'Invierno', 'Primavera']

# Eliminar columna "estación" si existe
inumet_vals = inumet_estacional.drop(columns='estación') if 'estación' in inumet_estacional.columns else inumet_estacional.copy()
chirps_vals = chirps_estacional.drop(columns='estación') if 'estación' in chirps_estacional.columns else chirps_estacional.copy()

# Estaciones meteorológicas y colores
estaciones_meteorologicas = inumet_vals.columns
colores = plt.cm.tab20.colors
colores_dict = {est: colores[i % len(colores)] for i, est in enumerate(estaciones_meteorologicas)}

# Crear handles para la leyenda
handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=colores_dict[est],
               markeredgecolor='black',
               markersize=10, label=est)
    for est in estaciones_meteorologicas
]

# Tamaños de fuente
title_size = 18
label_size = 15
tick_size = 13
legend_size = 13
r2_size = 12

# Iterar sobre cada estación solar
for i in range(4):
    x = inumet_vals.iloc[i]
    y = chirps_vals.iloc[i]

    # Eliminar valores NaN
    valid = x.notna() & y.notna()
    x = x[valid]
    y = y[valid]
    etiquetas = estaciones_meteorologicas[valid]

    # Datos para regresión
    X = x.values.reshape(-1, 1)
    Y = y.values
    model = LinearRegression()
    model.fit(X, Y)
    y_pred = model.predict(X)
    r2 = model.score(X, Y)
    corr, _ = pearsonr(x, y)

    # Dibujar puntos
    textos = []
    for xi, yi, etiqueta in zip(x, y, etiquetas):
        axs[i].scatter(xi, yi, color=colores_dict[etiqueta], s=80, edgecolor='black', linewidth=0.5)
        textos.append(
            axs[i].text(xi, yi, etiqueta, fontsize=tick_size, color=colores_dict[etiqueta],
                        weight='bold', alpha=0.9)
        )

    # Ajustar etiquetas automáticamente para que no se superpongan
    adjust_text(textos, ax=axs[i], arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    # Línea de regresión
    axs[i].plot(x, y_pred, color='red', linewidth=2)

    # Título de cada subplot con estación solar
    axs[i].set_title(nombres_estaciones_solares[i], fontsize=title_size, fontweight='bold')
    axs[i].grid(True, linestyle='--', alpha=0.7)
    axs[i].tick_params(axis='both', labelsize=tick_size)

    # Escalas dinámicas por plot
    margen = 30
    axs[i].set_xlim(x.min() - margen, x.max() + margen)
    axs[i].set_ylim(y.min() - margen, y.max() + margen)

    # Texto con R² y correlación
    axs[i].text(0.05, 0.95, f"$R^2$ = {r2:.2f}\nCorrelación = {corr:.2f}",
                fontsize=r2_size, style='italic', transform=axs[i].transAxes,
                verticalalignment='top', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

# Leyenda **debajo del eje X**, centrada, en dos filas
fig.legend(handles=handles, title="Estaciones Meteorológicas",
           loc='upper center', bbox_to_anchor=(0.5, -0.12),
           ncol=int(np.ceil(len(estaciones_meteorologicas)/2)),
           fontsize=legend_size, title_fontsize=legend_size)

# Títulos unificados de ejes
fig.text(0.5, 0.04, "INUMET (mm)", ha='center', fontsize=label_size, fontweight='bold')
fig.text(0.04, 0.5, "CHIRPS (mm)", va='center', rotation='vertical', fontsize=label_size, fontweight='bold')

# Título general
fig.suptitle("Dispersión Estacional INUMET - CHIRPS", fontsize=title_size, fontweight='bold')

plt.tight_layout(rect=[0.05, 0.1, 0.95, 0.96])
plt.savefig("dispersión_chirps_vs_inumet_estacional_labels_sin_superposicion.png", dpi=300, bbox_inches='tight')
plt.show()

##Análisis Anual de las variables

In [ ]:
# Cambiamos la frecuencia a anual sumando las precipitaciones de cada día
chirps_anual = chirps.sum()

chirps_anual

In [ ]:
import matplotlib.pyplot as plt

# Crear figura con 2 subplots (INUMET arriba, CHIRPS abajo)
fig, axs = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# INUMET
media_inumet = inumet_anual.values.mean()
axs[0].bar(inumet_anual.index, inumet_anual.values, color='skyblue', edgecolor='black')
axs[0].axhline(y=media_inumet, color='red', linestyle='--', linewidth=2,
               label=f'Media: {media_inumet:.2f} mm')
axs[0].set_title("Precipitación Acumulada Anual - INUMET", fontsize=18, fontweight='bold')
axs[0].set_ylabel("Precipitación (mm)", fontsize=16, fontweight='bold')
axs[0].legend(fontsize=14, loc='upper right')
axs[0].grid(axis='y', linestyle='--', alpha=0.7)

# CHIRPS
media_chirps = chirps_anual.values.mean()
axs[1].bar(chirps_anual.index, chirps_anual.values, color='skyblue', edgecolor='black')
axs[1].axhline(y=media_chirps, color='red', linestyle='--', linewidth=2,
               label=f'Media: {media_chirps:.2f} mm')
axs[1].set_title("Precipitación Acumulada Anual - CHIRPS", fontsize=18, fontweight='bold')
axs[1].set_ylabel("Precipitación (mm)", fontsize=16, fontweight='bold')
axs[1].set_xlabel("Estaciones", fontsize=16, fontweight='bold')
axs[1].legend(fontsize=14, loc='upper right')
axs[1].grid(axis='y', linestyle='--', alpha=0.7)

# Ajustar etiquetas del eje x más grandes
axs[1].tick_params(axis='x', rotation=45, labelsize=14)
axs[1].set_xticklabels(chirps_anual.index, ha='right', fontsize=14)

# Ajustar etiquetas del eje y también más grandes
axs[0].tick_params(axis='y', labelsize=14)
axs[1].tick_params(axis='y', labelsize=14)

# Ajuste de layout final
plt.tight_layout()
plt.savefig("serie_chirps_vs_inumet_anual2.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Nombres de las estaciones
estaciones = inumet_anual.index
x = np.arange(len(estaciones))  # posiciones en eje x
width = 0.35  # ancho de las barras

# Datos
val_inumet = inumet_anual.values
val_chirps = chirps_anual.reindex(estaciones).values  # asegurar mismo orden

# Medias
media_inumet = val_inumet.mean()
media_chirps = val_chirps.mean()

# Crear figura y eje
fig, ax = plt.subplots(figsize=(14, 7))

# Barras
barras_inumet = ax.bar(x - width/2, val_inumet, width, label='INUMET',
                       color='skyblue', edgecolor='black')
barras_chirps = ax.bar(x + width/2, val_chirps, width, label='CHIRPS',
                       color='orange', edgecolor='black')

# Líneas de promedio
ax.axhline(media_inumet, color='blue', linestyle='--', linewidth=2,
           label=f'Media INUMET: {media_inumet:.2f} mm')
ax.axhline(media_chirps, color='darkorange', linestyle='--', linewidth=2,
           label=f'Media CHIRPS: {media_chirps:.2f} mm')

# Personalización
ax.set_ylabel("Precipitación Acumulada Anual (mm)", fontsize=16, fontweight='bold')
ax.set_title("Precipitación Acumulada Anual INUMET - CHIRPS", fontsize=18, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(estaciones, rotation=45, ha='right', fontsize=14)
ax.set_xlabel("Estaciones", fontsize=16, fontweight='bold')
ax.tick_params(axis='y', labelsize=14)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.legend(fontsize=14, loc='upper right')

plt.tight_layout()
plt.savefig("serie_chirps_vs_inumet_anual.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

# Variables con datos anuales
x = inumet_anual.copy()
y = chirps_anual.copy()

# Eliminar valores NaN en ambos
valid = x.notna() & y.notna()
x = x[valid]
y = y[valid]

# Nombres para etiquetar
etiquetas = x.index

# Estaciones meteorológicas y colores
estaciones_meteorologicas = x.index
colores = plt.cm.tab20.colors
colores_dict = {est: colores[i % len(colores)] for i, est in enumerate(estaciones_meteorologicas)}

# Regresión lineal
X = x.values.reshape(-1, 1)
Y = y.values
model = LinearRegression()
model.fit(X, Y)
y_pred = model.predict(X)
r2 = model.score(X, Y)
corr, _ = pearsonr(x, y)

plt.figure(figsize=(10, 8))

# Scatter plot con color y borde
for xi, yi, etiqueta in zip(x, y, etiquetas):
    color = colores_dict.get(etiqueta, 'gray')
    plt.scatter(xi, yi, color=color, edgecolor='black', s=120, alpha=0.7)
    plt.text(xi + 8, yi + 6, etiqueta, fontsize=13, fontweight='bold',
             color=color, alpha=0.9)

# Línea de regresión
plt.plot(x, y_pred, color='red', linewidth=2, label='Línea de Tendencia')

# Ejes y título
plt.xlabel("INUMET (mm)", fontsize=14, fontweight='bold')
plt.ylabel("CHIRPS (mm)", fontsize=14, fontweight='bold')
plt.title("Dispersión Anual INUMET - CHIRPS", fontsize=18, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# Texto con R² y correlación
plt.text(0.05, 0.95,
         f"$R^2$ = {r2:.2f}\nCorrelación = {corr:.2f}",
         fontsize=12, style='italic', transform=plt.gca().transAxes,
         verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# --- Márgenes automáticos ---
margen_x = (x.max() - x.min()) * 0.1
margen_y = (y.max() - y.min()) * 0.1
plt.xlim(x.min() - margen_x, x.max() + margen_x)
plt.ylim(y.min() - margen_y, y.max() + margen_y)

plt.tight_layout()
plt.savefig("dispersion_anual_margen.png", dpi=300)
plt.show()

## Calculo de Errores

In [ ]:
# Renombrar columnas en chirps
columnas = {
    'paradorlavibora': 'lavibora',
    'elaguila': 'aguila'
}

# Aplicar a los DataFrames
chirps.rename(columns=columnas, inplace=True)
chirps_mensual.rename(columns=columnas, inplace=True)
chirps_estacional.rename(columns=columnas, inplace=True)

In [ ]:
# Filtrar columnas
columnas_seleccionadas = ['lavibora', 'piedrasola', 'sarandidelnavarro', 'sanjavier', 'chapicuy',
                          'piedrascoloradas', 'aguila', 'pueblogrecco', 'guichon', 'nuevoberlin', 'eleucaliptus']
inumet_diario_filtrado = inumet_diario[columnas_seleccionadas]
chirps_filtrado = chirps[columnas_seleccionadas]

# Renombrar las columnas para diferenciar la fuente de datos
inumet_diario_reno = inumet_diario_filtrado.add_suffix('_inumet')
chirps_reno = chirps_filtrado.add_suffix('_chirps')

# Concatenar ambos DataFrames
diarios = pd.concat([inumet_diario_reno, chirps_reno], axis=1)

# Mostrar el DataFrame resultante
diarios

In [ ]:
# Filtrar columnas
inumet_mensual_filtrado = inumet_mensual[columnas_seleccionadas]
chirps_mensual_filtrado = chirps_mensual[columnas_seleccionadas]

# Renombrar las columnas para diferenciar la fuente de datos
inumet_mensual_reno = inumet_mensual_filtrado.add_suffix('_inumet')
chirps_mensual_reno = chirps_mensual_filtrado.add_suffix('_chirps')

# Concatenar ambos DataFrames
mensual = pd.concat([inumet_mensual_reno, chirps_mensual_reno], axis=1)

# Mostrar el DataFrame resultante
mensual

In [ ]:
# Filtrar columnas
inumet_estacional_filtrado = inumet_estacional[columnas_seleccionadas]
chirps_estacional_filtrado = chirps_estacional[columnas_seleccionadas]

# Renombrar las columnas para diferenciar la fuente de datos
inumet_estacional_reno = inumet_estacional_filtrado.add_suffix('_inumet')
chirps_estacional_reno = chirps_estacional_filtrado.add_suffix('_chirps')

# Concatenar ambos DataFrames
estacional = pd.concat([inumet_estacional_reno, chirps_estacional_reno], axis=1)

# Mostrar el DataFrame resultante
estacional

In [ ]:
# Función para calcular las métricas de error y correlación
def calcular_metricas(df, columnas_inumet, columnas_chirps):
    resultados = {}

    for col_inumet, col_chirps in zip(columnas_inumet, columnas_chirps):
        y_true = df[col_inumet].dropna()
        y_pred = df[col_chirps].dropna()

        # Asegurarse de que tengan el mismo tamaño tras eliminar NaN
        common_index = y_true.index.intersection(y_pred.index)
        y_true = y_true.loc[common_index]
        y_pred = y_pred.loc[common_index]

        # Cálculos de métricas
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        mpe = np.mean((y_true - y_pred) / y_true) * 100
        bias = np.mean(y_pred - y_true)
        r = np.corrcoef(y_true, y_pred)[0, 1]
        r2 = r2_score(y_true, y_pred)

        # Guardar resultados
        resultados[col_inumet.replace('_inumet', '')] = {
            'RMSE': rmse,
            'MSE': mse,
            'MAE': mae,
            'MPE': mpe,
            'TASA BIAS': bias,
            'R': r,
            'R2': r2
        }

    return resultados

# Identificar columnas de cada fuente
columnas_inumet = [col for col in diarios.columns if col.endswith('_inumet')]
columnas_chirps = [col for col in diarios.columns if col.endswith('_chirps')]

# Calcular métricas para los 3 DataFrames
resultados_diarios = calcular_metricas(diarios, columnas_inumet, columnas_chirps)
resultados_mensual = calcular_metricas(mensual, columnas_inumet, columnas_chirps)
resultados_estacional = calcular_metricas(estacional, columnas_inumet, columnas_chirps)

# Mostrar resultados

df_resultados_diarios = pd.DataFrame(resultados_diarios).T
df_resultados_mensual = pd.DataFrame(resultados_mensual).T
df_resultados_estacional = pd.DataFrame(resultados_estacional).T

In [ ]:
df_resultados_diarios

In [ ]:
df_resultados_mensual

In [ ]:
df_resultados_estacional